# RSNA Knee — pseudo-label training, Phase 1 tasks 1–4

1. DICOM loader  2. five-slice 2.5D dataset  3. EfficientNetV2-S baseline  4. two-stage training pipeline. Reports/LLM are used offline to create `pseudo_labels.csv`; this notebook consumes confidence-filtered pseudo-labels without calling an external API.

In [ ]:
from pathlib import Path
import gc, json, random, time, warnings
import cv2, numpy as np, pandas as pd, pydicom
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
SEED=2026; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGETS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
COMP=Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
TRAIN_CSV=COMP/'train.csv'; SERIES_CSV=COMP/'train_series.csv'; DICOM_ROOT=COMP/'train_series'
PSEUDO_CANDIDATES=[p for r in Path('/kaggle/input').iterdir() if r.is_dir() and r.name!='competitions' for p in (r/'pseudo_labels.csv',r/'data'/'pseudo_labels.csv')]
PSEUDO_CSV=next((p for p in PSEUDO_CANDIDATES if p.is_file()),None)
if PSEUDO_CSV is None: raise FileNotFoundError('Attach pseudo_labels.csv as a Kaggle Dataset')
for p in (TRAIN_CSV,SERIES_CSV,DICOM_ROOT):
    if not p.exists(): raise FileNotFoundError(p)
CFG=dict(image_size=384,num_slices=5,batch_size=4,num_workers=2,epochs=50,freeze_epochs=5,lr=2e-4,weight_decay=1e-4,amp=True)
print(DEVICE,PSEUDO_CSV,CFG)

## Task 1 — DICOM loader

In [ ]:
def position(ds,plane):
    ipp=getattr(ds,'ImagePositionPatient',None); axis={'Sagittal':0,'Coronal':1,'Axial':2}.get(plane,2)
    if ipp is not None and len(ipp)>=3: return float(ipp[axis])
    return float(getattr(ds,'SliceLocation',getattr(ds,'InstanceNumber',0)))
def load_volume(path,plane,size=384):
    files=list(Path(path).glob('*.dcm'))
    if not files: raise FileNotFoundError(path)
    records=[]
    for f in files:
        ds=pydicom.dcmread(str(f),force=True); records.append((position(ds,plane),ds))
    out=[]
    for _,ds in sorted(records,key=lambda z:z[0]):
        x=ds.pixel_array.astype(np.float32)*float(getattr(ds,'RescaleSlope',1))+float(getattr(ds,'RescaleIntercept',0))
        if str(getattr(ds,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        finite=x[np.isfinite(x)]; lo,hi=np.percentile(finite,[.5,99.5]); x=np.zeros_like(x) if hi<=lo else np.clip((x-lo)/(hi-lo),0,1)
        out.append(cv2.resize(x,(size,size),interpolation=cv2.INTER_AREA))
    return np.stack(out).astype(np.float32)
def stack5(volume,center):
    ids=[min(max(center+i,0),len(volume)-1) for i in (-2,-1,0,1,2)]
    return volume[ids]

## Task 2 — confidence-filtered pseudo-labels and 2.5D Dataset

In [ ]:
meta=pd.read_csv(TRAIN_CSV); series=pd.read_csv(SERIES_CSV); pseudo=pd.read_csv(PSEUDO_CSV)
required={'StudyInstanceUID','SeriesInstanceUID','Anatomical_Plane'}
if missing:=required-set(series.columns): raise ValueError(f'train_series missing {missing}')
for c in TARGETS:
    if f'pred_{c}' not in pseudo or f'conf_{c}' not in pseudo: raise ValueError(f'pseudo label columns missing for {c}')
high=np.logical_and.reduce([pseudo[f'conf_{c}'].eq('HIGH').to_numpy() for c in TARGETS])
labels=pseudo.loc[high,['StudyInstanceUID']].copy()
for c in TARGETS: labels[c]=pseudo.loc[high,f'pred_{c}'].astype(np.float32).to_numpy()
gold_mask=meta[TARGETS].notna().all(axis=1); gold=meta.loc[gold_mask,['StudyInstanceUID',*TARGETS]].copy()
train_frame=series.merge(labels,on='StudyInstanceUID',how='inner'); val_frame=series.merge(gold,on='StudyInstanceUID',how='inner')
if set(train_frame.StudyInstanceUID)&set(val_frame.StudyInstanceUID): train_frame=train_frame[~train_frame.StudyInstanceUID.isin(set(val_frame.StudyInstanceUID))]
class Knee25D(Dataset):
    def __init__(self,frame,training): self.df=frame.reset_index(drop=True); self.training=training
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; uid,sid=str(r.StudyInstanceUID),str(r.SeriesInstanceUID); v=load_volume(DICOM_ROOT/uid/sid,str(r.Anatomical_Plane),CFG['image_size']); center=np.random.randint(len(v)) if self.training else len(v)//2
        return {'image':torch.from_numpy(stack5(v,center).copy()),'target':torch.tensor(r[TARGETS].to_numpy(np.float32)),'study_uid':uid}
train_ds,val_ds=Knee25D(train_frame,True),Knee25D(val_frame,False)
kw=dict(batch_size=CFG['batch_size'],num_workers=CFG['num_workers'],pin_memory=DEVICE.type=='cuda')
train_loader=DataLoader(train_ds,shuffle=True,**kw); val_loader=DataLoader(val_ds,shuffle=False,**kw)
print('HIGH pseudo studies',labels.StudyInstanceUID.nunique(),'train series',len(train_ds),'gold val studies',val_frame.StudyInstanceUID.nunique())

## Task 3 — EfficientNetV2-S 5-channel baseline

In [ ]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
class EfficientNet25D(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__(); m=efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT if pretrained else None); old=m.features[0][0]; new=nn.Conv2d(5,old.out_channels,old.kernel_size,old.stride,old.padding,bias=False)
        if pretrained:
            with torch.no_grad(): new.weight.copy_(old.weight.mean(1,keepdim=True).repeat(1,5,1,1)*3/5)
        m.features[0][0]=new; m.classifier=nn.Identity(); self.backbone=m; self.classifier=nn.Sequential(nn.LayerNorm(1280),nn.Dropout(.3),nn.Linear(1280,12))
    def forward(self,x): return self.classifier(self.backbone(x))
    def freeze(self): self.backbone.requires_grad_(False)
    def unfreeze(self): self.backbone.requires_grad_(True)
class FocalBCE(nn.Module):
    def __init__(self,gamma=2,alpha=.25): super().__init__(); self.g,self.a=gamma,alpha
    def forward(self,z,y):
        b=nn.functional.binary_cross_entropy_with_logits(z,y,reduction='none'); p=torch.sigmoid(z); pt=p*y+(1-p)*(1-y); at=self.a*y+(1-self.a)*(1-y); return (at*(1-pt).pow(self.g)*b).mean()
model=EfficientNet25D(True).to(DEVICE); model.freeze(); criterion=FocalBCE()
print('parameters',sum(p.numel() for p in model.parameters()),'trainable',sum(p.numel() for p in model.parameters() if p.requires_grad))

## Task 4 — frozen 5 epochs, then full fine-tuning

In [ ]:
from collections import defaultdict
def macro_auc(y,p):
    scores=[roc_auc_score(y[:,i],p[:,i]) for i in range(y.shape[1]) if np.unique(y[:,i]).size==2]; return float(np.mean(scores)) if scores else float('nan')
@torch.no_grad()
def validate():
    model.eval(); logits=defaultdict(list); targets={}
    for b in val_loader:
        z=model(b['image'].to(DEVICE)).float().cpu().numpy()
        for uid,zz,yy in zip(b['study_uid'],z,b['target'].numpy()): logits[uid].append(zz); targets[uid]=yy
    u=sorted(logits); z=np.stack([np.mean(logits[k],0) for k in u]); y=np.stack([targets[k] for k in u]); return macro_auc(y,1/(1+np.exp(-np.clip(z,-30,30))))
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay']); scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=2); use_amp=CFG['amp'] and DEVICE.type=='cuda'; scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
best=-1.; history=[]; out=Path('/kaggle/working/checkpoints'); out.mkdir(parents=True,exist_ok=True)
for epoch in range(CFG['epochs']):
    if epoch==CFG['freeze_epochs']: model.unfreeze()
    model.train(); total=n=0
    for b in tqdm(train_loader,desc=f'Epoch {epoch+1}/{CFG["epochs"]}'):
        x,y=b['image'].to(DEVICE),b['target'].to(DEVICE); optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16,enabled=use_amp): loss=criterion(model(x),y)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update(); total+=loss.item()*len(x); n+=len(x)
    score=validate(); scheduler.step(epoch+1); history.append({'epoch':epoch+1,'loss':total/max(n,1),'macro_auc':score}); print(history[-1])
    if np.isfinite(score) and score>best:
        best=score; torch.save({'model_state_dict':model.state_dict(),'epoch':epoch+1,'val_macro_auc':score,'target_columns':TARGETS},out/'efficientnet_best.pth')
    (Path('/kaggle/working')/'training_history.json').write_text(json.dumps(history,indent=2))
print('best macro AUC',best,'checkpoint',out/'efficientnet_best.pth')